# OutputFixingParser - 파싱 오류를 LLM이 자동 수정해요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `OutputFixingParser`의 동작 원리(파싱 실패 → LLM 재요청 → 재파싱)를 설명할 수 있어요
- 기존 파서를 `OutputFixingParser`로 감싸서 오류 복구 기능을 추가할 수 있어요
- 필드 누락, 타입 불일치, JSON 형식 오류 등 다양한 파싱 오류를 자동으로 처리할 수 있어요

## 사전 지식

- **래퍼 패턴(Wrapper Pattern)**: 기존 객체를 새 객체로 감싸서 추가 기능을 부여하는 설계 패턴이에요. `OutputFixingParser.from_llm(parser=기존파서, llm=모델)` 형태로 사용해요
- **OutputParserException**: LangChain이 파싱 실패 시 발생시키는 예외 클래스예요. `OutputFixingParser`는 이 예외를 잡아서 LLM에 수정을 요청해요

## 파싱 실패 → 자동 수정 흐름

```mermaid
flowchart TD
    A["LLM 응답<br/>(잘못된 형식)"] -->|"파싱 시도"| B["기본 파서<br/>(PydanticOutputParser 등)"]
    B -->|"파싱 성공"| C["정상 결과 반환"]
    B -->|"OutputParserException 발생"| D["OutputFixingParser<br/>오류 메시지 수집"]
    D -->|"오류 + 원본 출력 전달"| E["LLM<br/>(수정 요청)"]
    E -->|"수정된 JSON 반환"| F["기본 파서<br/>(재파싱)"]
    F -->|"파싱 성공"| C
    F -->|"재파싱도 실패"| G["Exception 발생<br/>(수동 처리 필요)"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,D,E,F process
    class C output
    class G error
```

## 처리할 수 있는 오류 유형

| 오류 유형 | 예시 | 자동 수정 가능 |
|-----------|------|----------------|
| Python dict 형식 | `{'key': 'value'}` (작은따옴표) | 가능 |
| 필드 누락 | `stock` 필드 없음 | 가능 (LLM이 추론해서 채움) |
| 타입 불일치 | `price: "삼만원"` (숫자 대신 문자) | 가능 |
| 키 없는 JSON | `{name: "값"}` (따옴표 없는 키) | 가능 |
| 스키마 완전 무시 | 전혀 다른 형식 | 불가능 (LLM도 해결 못할 수 있음) |

> **실무 팁**: `OutputFixingParser`는 추가 LLM 호출을 한 번 더 하므로 비용과 지연이 늘어요. 프롬프트를 잘 작성해서 오류 발생 자체를 줄이고, `OutputFixingParser`는 안전망으로만 사용하는 전략을 권장해요.

In [ ]:
from dotenv import load_dotenv

load_dotenv()


## 기본 사용법: 잘못된 형식 수정하기

먼저 파싱이 실패하는 상황을 만들고, `OutputFixingParser`로 해결해봅시다.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# ---------------------------------------------------
# Pydantic 모델 정의 및 기본 파서 생성
# ---------------------------------------------------

# 1단계: Pydantic 모델 정의
class MusicAlbum(BaseModel):
    """음악 앨범 정보"""
    
    artist: str = Field(description="아티스트 이름")
    album_title: str = Field(description="앨범 제목")
    release_year: int = Field(description="발매 연도")
    genres: List[str] = Field(description="장르 목록")

# 2단계: 기본 파서 생성
# PydanticOutputParser: 정상적인 JSON만 파싱 가능 (오류 복구 없음)
base_parser = PydanticOutputParser(pydantic_object=MusicAlbum)

# 3단계: 잘못된 형식의 출력 예시 (일부러 오류 유발)
# Python dict 형식 (작은따옴표): 표준 JSON은 큰따옴표를 사용
misformatted_output = "{'artist': 'BTS', 'album_title': 'MAP OF THE SOUL: 7', 'release_year': 2020, 'genres': ['K-Pop', 'Hip Hop']}"

print("=" * 60)
print("❌ 잘못된 형식의 출력")
print("=" * 60)
print(misformatted_output)
print()

In [ ]:
# ---------------------------------------------------
# 기본 파서로 파싱 시도 - OutputParserException 확인
# ---------------------------------------------------

print("=" * 60)
print("🔴 기본 파서로 파싱 시도")
print("=" * 60)
try:
    result = base_parser.parse(misformatted_output)
    print("파싱 성공:", result)
except Exception as e:
    # OutputParserException: LangChain이 파싱 실패 시 발생시키는 예외
    print(f"파싱 실패: {type(e).__name__}")
    print(f"오류 메시지: {str(e)[:100]}...")
    print()

In [ ]:
# OutputFixingParser로 파싱 (자동 수정)
from langchain.output_parsers import OutputFixingParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# OutputFixingParser 생성 - 기본 파서와 LLM을 감싸서 생성
# TODO: OutputFixingParser.from_llm()으로 fixing_parser를 생성하세요


print("=" * 60)
print("✅ OutputFixingParser로 파싱 시도")
print("=" * 60)
# TODO: fixing_parser.parse()로 misformatted_output을 파싱하세요


---

## 실용 예제: LCEL 체인에 OutputFixingParser 통합해요

개별 `parse()` 호출 외에도 LCEL 체인의 파서 자리에 `OutputFixingParser`를 바로 꽂을 수 있어요. 체인 전체가 더 안정적으로 동작해요.

> **실무 팁**: `prompt | model | fixing_parser` 형태로 체인을 구성하면, 프로덕션에서 간헐적으로 발생하는 파싱 오류를 조용히 복구할 수 있어요. 다만 복구 시도 횟수와 비용을 모니터링하세요.

In [ ]:
# ---------------------------------------------------
# LCEL 체인에 OutputFixingParser 통합 - 안정성 향상
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

# 영화 정보 모델
class MovieInfo(BaseModel):
    """영화 정보"""
    
    title: str = Field(description="영화 제목")
    director: str = Field(description="감독 이름")
    year: int = Field(description="개봉 연도")
    rating: float = Field(description="평점 (0.0-10.0)", ge=0.0, le=10.0)
    plot_summary: str = Field(description="줄거리 요약 (2-3문장)")

# 1단계: 기본 파서와 OutputFixingParser 생성
movie_parser = PydanticOutputParser(pydantic_object=MovieInfo)
# OutputFixingParser.from_llm(): 기본 파서 + LLM을 래퍼(Wrapper)로 감싸기
# 파싱 실패 시 LLM에 "이 출력을 올바른 형식으로 수정해줘"라고 재요청
# TODO: OutputFixingParser.from_llm()으로 movie_fixing_parser를 생성하세요


# 2단계: 프롬프트 구성
# TODO: ChatPromptTemplate.from_messages()로 프롬프트를 구성하세요


# TODO: partial()로 format_instructions를 주입하세요


# 3단계: 체인 구성 - OutputFixingParser를 파서 자리에 바로 사용
# 파싱 오류가 발생해도 LLM이 자동 수정 후 재파싱
# TODO: prompt | model | movie_fixing_parser 로 체인을 구성하세요


# 실행
# TODO: chain.invoke()로 "기생충" 영화 정보를 추출하세요


print("\n" + "=" * 60)
print("🎬 영화 정보 추출 (안정적)")
print("=" * 60)
# TODO: 결과의 제목, 감독, 개봉 연도, 평점, 줄거리를 출력하세요


---

## 다양한 오류 케이스: 자동 수정 범위를 확인해요

실제로 발생할 수 있는 3가지 오류 케이스를 직접 실험해봐요. 각각 기본 파서 실패 → `OutputFixingParser` 복구 과정을 눈으로 확인할 수 있어요.

> **주의**: 필드 누락의 경우 LLM이 문맥을 추론해서 기본값을 채워 넣어요. 이 값이 항상 정확하다고 보장할 수 없으므로, 결과를 검토하는 로직을 추가하는 편이 안전해요.

In [ ]:
# ---------------------------------------------------
# 다양한 오류 유형별 자동 수정 실험
# ---------------------------------------------------

# 제품 정보 모델
class Product(BaseModel):
    """제품 정보"""
    
    name: str = Field(description="제품명")
    price: int = Field(description="가격 (원)")
    stock: int = Field(description="재고 수량")
    
# 파서 생성
product_parser = PydanticOutputParser(pydantic_object=Product)
# TODO: OutputFixingParser.from_llm()으로 product_fixing_parser를 생성하세요


# 다양한 오류 케이스
error_cases = [
    {
        "name": "필드 누락",
        "data": '{"name": "노트북", "price": 1200000}'  # stock 필드 누락
    },
    {
        "name": "타입 불일치",
        "data": '{"name": "마우스", "price": "삼만원", "stock": 50}'  # price가 숫자 아닌 문자열
    },
    {
        "name": "잘못된 JSON",
        "data": "{name: '키보드', price: 80000, stock: 30}"  # 키에 따옴표 없음
    },
]

print("\n" + "=" * 60)
print("🔧 다양한 오류 케이스 자동 수정")
print("=" * 60)

for case in error_cases:
    print(f"\n[{case['name']}]")
    print(f"입력: {case['data']}")
    
    # 기본 파서로 시도
    try:
        product_parser.parse(case['data'])
        print("✅ 기본 파서로 파싱 성공")
    except:
        print("❌ 기본 파서 실패")
        
    # OutputFixingParser로 시도
    # 오류 발생 시: 오류 메시지 + 원본 출력 → LLM에 수정 요청 → 재파싱
    # TODO: product_fixing_parser.parse()로 자동 수정을 시도하세요


## 연습 문제

다음 연습 문제를 통해 `OutputFixingParser`의 활용법을 직접 실습해봅시다.

### 연습 1: 잘못된 주소록 데이터 수정하기

`OutputFixingParser`를 사용하여 형식이 잘못된 주소록 데이터를 자동으로 수정해보세요.

**요구사항:**
- Pydantic 모델 `Contact`를 정의하세요:
  - `name` (str): 이름
  - `phone` (str): 전화번호
  - `email` (str): 이메일 주소
  - `city` (str): 거주 도시
- 기본 `PydanticOutputParser`로 아래 잘못된 데이터를 파싱하면 실패하는 것을 확인하세요
- `OutputFixingParser`로 동일한 데이터를 성공적으로 파싱하세요

잘못된 데이터 3건:
```python
# 작은따옴표 사용 (JSON 규격 위반)
"{'name': '홍길동', 'phone': '010-1234-5678', 'email': 'hong@email.com', 'city': '서울'}"

# 필드명 오류 + 필드 누락
'{"이름": "김영희", "연락처": "010-9876-5432", "이메일": "kim@email.com"}'

# 따옴표 없는 키
'{name: "이철수", phone: "010-5555-1234", email: "lee@email.com", city: "부산"}'
```

In [ ]:
# ============================================================
# TODO: 잘못된 주소록 데이터 자동 수정하기
# 힌트: OutputFixingParser.from_llm(parser=base_parser, llm=model)로 생성하고
#       fixing_parser.parse()로 각 데이터를 파싱하세요
# 예상 결과: 3가지 오류 케이스 모두 자동 수정되어 Contact 객체로 반환됨
# ============================================================


### 연습 2: OutputFixingParser를 체인에 통합하기

`OutputFixingParser`를 LCEL 체인에 통합하여, LLM 출력이 불안정하더라도 안정적으로 파싱되는 체인을 만들어보세요.

**요구사항:**
- Pydantic 모델 `RecipeInfo`를 정의하세요:
  - `name` (str): 요리 이름
  - `ingredients` (List[str]): 주요 재료 (3-5개)
  - `cooking_time_minutes` (int): 조리 시간 (분)
  - `difficulty` (str): 난이도 (쉬움/보통/어려움)
- 기본 `PydanticOutputParser`를 `OutputFixingParser`로 감싸서 체인에 사용하세요
- `ChatPromptTemplate`으로 프롬프트를 구성하세요
- 테스트: "김치찌개" 레시피 정보를 추출하세요

In [ ]:
# ============================================================
# TODO: OutputFixingParser를 LCEL 체인에 통합하기
# 힌트: base_parser로 format_instructions를 생성하고,
#       체인의 파서 자리에 fixing_parser를 사용하세요
# 예상 결과: 파싱 오류가 발생해도 자동 수정되어 RecipeInfo 객체로 반환됨
# ============================================================


## 핵심 요약

| 항목 | 설명 |
|------|------|
| **역할** | 파싱 실패 시 LLM에 수정을 요청하고 재파싱 |
| **생성 방법** | `OutputFixingParser.from_llm(parser=기존파서, llm=모델)` |
| **추가 비용** | 오류 발생 시 LLM 호출 1회 추가 |
| **한계** | 원본 데이터와 무관한 필드는 LLM이 임의 추론할 수 있음 |

## OutputParser 시리즈 전체 선택 가이드

```mermaid
flowchart TD
    START["OutputParser 선택"] --> Q1{"출력 형태는?"}
    Q1 -->|"단순 텍스트"| A1["StrOutputParser<br/>노트북 01"]
    Q1 -->|"구조화된 데이터"| Q2{"타입 검증<br/>필요 여부"}
    Q1 -->|"쉼표 구분 목록"| A4["CommaSeparatedList<br/>노트북 03"]
    Q1 -->|"DataFrame 조회"| A5["PandasDataFrame<br/>노트북 05"]
    Q2 -->|"강력한 검증 필요"| Q3{"datetime / Enum<br/>포함 여부"}
    Q2 -->|"빠른 프로토타이핑"| A3["JsonOutputParser<br/>노트북 02"]
    Q3 -->|"예"| A2a["PydanticOutputParser<br/>+ datetime/Enum<br/>노트북 04"]
    Q3 -->|"아니오"| A2b["PydanticOutputParser<br/>노트북 02"]
    A1 --> FIX{"파싱 오류 발생?"}
    A2a --> FIX
    A2b --> FIX
    A3 --> FIX
    A4 --> FIX
    FIX -->|"예"| FIXING["OutputFixingParser로 감싸기<br/>노트북 06"]
    FIX -->|"아니오"| DONE["완료"]
    FIXING --> DONE

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class START input
    class Q1,Q2,Q3,FIX process
    class A1,A2a,A2b,A3,A4,A5,DONE output
    class FIXING error
```

## OutputParser 시리즈 마무리

축하해요! OutputParser 시리즈 6개 노트북을 모두 완료했어요. 정리하면 다음과 같아요.

| 노트북 | 파서 | 출력 타입 | 핵심 키워드 |
|--------|------|-----------|-------------|
| 01 | StrOutputParser | `str` | 텍스트, 스트리밍 |
| 02 | PydanticOutputParser / JsonOutputParser | Pydantic 객체 / `dict` | 타입 검증, 스키마 |
| 03 | CommaSeparatedListOutputParser | `list` | 목록, 키워드 추출 |
| 04 | PydanticOutputParser + datetime/Enum | `datetime` / Enum | 날짜 추출, 분류 |
| 05 | PandasDataFrameOutputParser | `Series` / `dict` | 자연어 데이터 조회 |
| 06 | OutputFixingParser | (기본 파서와 동일) | 오류 복구, 안정성 |

다음 단계로는 **03_Chains** 챕터에서 배운 파서들을 더 복잡한 체인과 에이전트에 통합하는 방법을 배워요.